# Projeto 2 - Computacao Grafica

## Felipe da Costa Coqueiro, NUSP: 11781361
## Fernando Alee Suaiden, NUSP: 12680836 

Cena 3D: **balada urbana com pista de danca e area externa**.

A cena foi ajustada para atender diretamente ao enunciado do Projeto 2: existe um ambiente interno (clube fechado com pista, palco e objetos) e um ambiente externo (rua/estacionamento com entrada metalica, postes, carro, lixeira e pessoas). Cada item principal vem de um arquivo `.obj` separado, usa material/textura, e e posicionado com transformacoes `model`. O codigo do projeto esta neste notebook: shaders, carregador Wavefront, transformacoes Model/View/Projection, camera, teclado e loop de renderizacao. Fora dele existem apenas arquivos de dados da cena (`.obj`, `.mtl` e imagens de textura prontas), alem do `README.md`.

## 1. Dependencias e caminhos

Usamos apenas OpenGL/PyOpenGL, GLFW para a janela, NumPy/PyGLM para matrizes e Pillow para carregar imagens de textura. Os modelos estao em arquivos Wavefront (`.obj`) separados, cada um com material (`.mtl`) e textura (`map_Kd`).

In [46]:
from pathlib import Path
from dataclasses import dataclass
import ctypes
import math

import glfw
from OpenGL.GL import *
import glm
import numpy as np
from PIL import Image

def find_project_root():
    """Localiza a pasta que contem assets/models mesmo se o notebook for aberto pela raiz do workspace."""
    search_roots = [Path.cwd(), *Path.cwd().parents]
    candidates = []
    for root in search_roots:
        candidates.append(root)
        candidates.append(root / "ComputacaoGrafica")

    for candidate in candidates:
        if (candidate / "assets" / "models").is_dir() and (candidate / "assets" / "textures").is_dir():
            return candidate.resolve()

    raise FileNotFoundError("Nao encontrei a pasta assets/models do projeto. Abra o notebook dentro da pasta ComputacaoGrafica ou do workspace clonado.")

BASE_DIR = find_project_root()
ASSETS_DIR = BASE_DIR / "assets"
MODELS_DIR = ASSETS_DIR / "models"
TEXTURES_DIR = ASSETS_DIR / "textures"

# Texturas de imagem usadas quando um material auxiliar do .mtl nao possui map_Kd proprio.
# Assim, os modelos continuam texturizados sem recorrer a iluminacao ou cor fixa.
MODEL_TEXTURE_FALLBACKS = {
    "bar_chair_round_01": TEXTURES_DIR / "bar_chair_round_01_bar_chair_round_01_diff_1k.jpg",
    "boombox": TEXTURES_DIR / "boombox_boombox_diff_1k.jpg",
    "club_room": TEXTURES_DIR / "club_ceiling_square_tiles_03_diff_1k.jpg",
    "disco_support": TEXTURES_DIR / "disco_metal_plate_diff_1k.jpg",
    "lounge_floor": TEXTURES_DIR / "club_accent_painted_concrete_diff_1k.jpg",
    "modern_ceiling_lamp_01": TEXTURES_DIR / "modern_ceiling_lamp_01_modern_ceiling_lamp_01_diff_1k.jpg",
    "parking_lot": TEXTURES_DIR / "parking_asphalt_07_diff_1k.jpg",
    "party_person_male_01": MODELS_DIR / "CMan0010.tif",
    "rollershutter_door": TEXTURES_DIR / "rollershutter_door_rollershutter_door_diff_1k.jpg",
    "sofa_01": TEXTURES_DIR / "sofa_01_Sofa_01_diff_1k.jpg",
    "street_lamp_01": TEXTURES_DIR / "street_lamp_01_street_lamp_01_diff_1k.jpg",
}

print("Pasta base:", BASE_DIR)
print("Pasta de modelos:", MODELS_DIR)
print("Pasta de texturas:", TEXTURES_DIR)

Pasta base: /home/coqzieiro/Área de Trabalho/comp-grafica-2/ComputacaoGrafica
Pasta de modelos: /home/coqzieiro/Área de Trabalho/comp-grafica-2/ComputacaoGrafica/assets/models
Pasta de texturas: /home/coqzieiro/Área de Trabalho/comp-grafica-2/ComputacaoGrafica/assets/textures


## 2. Shader moderno

O projeto nao usa funcoes depreciadas como `glTranslate`, `glRotate`, `glScale`, `glBegin`, `glEnd`, iluminacao fixa ou matrizes antigas do OpenGL. As matrizes `model`, `view` e `projection` sao calculadas no Python e enviadas como `uniform` para o shader.

In [47]:
VERTEX_SHADER_SOURCE = """
#version 330 core
layout (location = 0) in vec3 position;
layout (location = 1) in vec2 texcoord;

uniform mat4 model;
uniform mat4 view;
uniform mat4 projection;

out vec2 uv;

void main() {
    gl_Position = projection * view * model * vec4(position, 1.0);
    uv = texcoord;
}
"""

FRAGMENT_SHADER_SOURCE = """
#version 330 core
in vec2 uv;
out vec4 FragColor;

uniform sampler2D imageTexture;
uniform bool useSolidColor;
uniform vec4 solidColor;

void main() {
    if (useSolidColor) {
        FragColor = solidColor;
    } else {
        FragColor = texture(imageTexture, uv);
    }
}
"""

class ShaderProgram:
    def __init__(self, vertex_source, fragment_source):
        vertex = self._compile(GL_VERTEX_SHADER, vertex_source)
        fragment = self._compile(GL_FRAGMENT_SHADER, fragment_source)
        self.program = glCreateProgram()
        glAttachShader(self.program, vertex)
        glAttachShader(self.program, fragment)
        glLinkProgram(self.program)
        self._check_program(self.program)
        glDeleteShader(vertex)
        glDeleteShader(fragment)

    def use(self):
        glUseProgram(self.program)

    def uniform(self, name):
        return glGetUniformLocation(self.program, name)

    @staticmethod
    def _compile(shader_type, source):
        shader = glCreateShader(shader_type)
        glShaderSource(shader, source)
        glCompileShader(shader)
        ok = glGetShaderiv(shader, GL_COMPILE_STATUS)
        if not ok:
            log = glGetShaderInfoLog(shader).decode("utf-8", errors="replace")
            raise RuntimeError(f"Erro ao compilar shader:\n{log}")
        return shader

    @staticmethod
    def _check_program(program):
        ok = glGetProgramiv(program, GL_LINK_STATUS)
        if not ok:
            log = glGetProgramInfoLog(program).decode("utf-8", errors="replace")
            raise RuntimeError(f"Erro ao linkar programa:\n{log}")

## 3. Carregador Wavefront OBJ/MTL

Esta celula implementa um parser simples para `v`, `vt`, `f`, `mtllib` e `usemtl`. Faces com mais de tres vertices sao trianguladas por fan triangulation, o que permite importar modelos que nao estejam triangulados. O carregador prioriza `map_Kd` dos arquivos `.mtl`; quando algum submaterial auxiliar vem sem `map_Kd`, ele recebe uma textura de imagem equivalente do proprio asset ou uma textura de fallback definida no catalogo. Assim os modelos exibidos permanecem texturizados, sem efeitos de iluminacao e sem depender de cores fixas.

In [48]:
@dataclass
class MeshSegment:
    start: int
    count: int
    material: str
    texture_path: Path | None
    diffuse_color: tuple[float, float, float]

@dataclass
class CpuMesh:
    name: str
    vertex_data: np.ndarray
    segments: list

@dataclass
class GpuMesh:
    name: str
    vao: int
    vbo: int
    segments: list
    vertex_count: int

def resolve_obj_index(index_text, total):
    index = int(index_text)
    return index - 1 if index > 0 else total + index

def resolve_texture_path(mtl_path, value):
    tokens = value.split()
    if not tokens:
        return None

    # map_Kd pode vir com opcoes como "-s 1 1 1 textura.png".
    # O projeto usa caminhos simples, mas essa regra deixa o parser mais seguro.
    texture_name = tokens[-1]
    texture_path = (mtl_path.parent / texture_name).resolve()
    return texture_path

def parse_mtl(mtl_path, model_name=None):
    materials = {}
    current = None
    for raw_line in mtl_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split(maxsplit=1)
        command = parts[0]
        value = parts[1] if len(parts) > 1 else ""
        if command == "newmtl":
            current = value.strip()
            materials[current] = {"texture_path": None, "diffuse_color": (1.0, 1.0, 1.0)}
        elif command == "Kd" and current:
            rgb = tuple(float(x) for x in value.split()[:3])
            materials[current]["diffuse_color"] = rgb
        elif command == "map_Kd" and current:
            materials[current]["texture_path"] = resolve_texture_path(mtl_path, value)

    fallback_texture = MODEL_TEXTURE_FALLBACKS.get(model_name)
    first_texture = next((info["texture_path"] for info in materials.values() if info.get("texture_path")), None)
    for info in materials.values():
        if info.get("texture_path") is None:
            info["texture_path"] = fallback_texture or first_texture

    return materials

def load_obj_mesh(obj_path):
    obj_path = Path(obj_path)
    positions = []
    texcoords = []
    out_positions = []
    out_texcoords = []
    segments = []
    material_info = {}
    current_material = None
    active_segment_material = None
    active_segment_start = 0

    def close_segment():
        nonlocal active_segment_material, active_segment_start
        end = len(out_positions)
        if active_segment_material is not None and end > active_segment_start:
            info = material_info.get(active_segment_material, {"texture_path": MODEL_TEXTURE_FALLBACKS.get(obj_path.stem), "diffuse_color": (1.0, 1.0, 1.0)})
            segments.append(MeshSegment(
                active_segment_start,
                end - active_segment_start,
                active_segment_material,
                info.get("texture_path"),
                tuple(info.get("diffuse_color", (1.0, 1.0, 1.0))),
            ))
        active_segment_material = None
        active_segment_start = end

    def use_material(material_name):
        nonlocal active_segment_material, active_segment_start
        if active_segment_material != material_name:
            close_segment()
            active_segment_material = material_name
            active_segment_start = len(out_positions)

    for raw_line in obj_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        command = parts[0]

        if command == "mtllib":
            mtl_path = obj_path.parent / " ".join(parts[1:])
            material_info.update(parse_mtl(mtl_path, obj_path.stem))
        elif command == "usemtl":
            current_material = " ".join(parts[1:])
        elif command == "v":
            positions.append(tuple(map(float, parts[1:4])))
        elif command == "vt":
            texcoords.append(tuple(map(float, parts[1:3])))
        elif command == "f":
            if current_material is None:
                raise ValueError(f"Face sem usemtl em {obj_path.name}.")
            use_material(current_material)
            polygon = []
            for token in parts[1:]:
                values = token.split("/")
                vi = resolve_obj_index(values[0], len(positions))
                ti = None
                if len(values) > 1 and values[1]:
                    ti = resolve_obj_index(values[1], len(texcoords))
                polygon.append((vi, ti))

            for i in range(1, len(polygon) - 1):
                for vi, ti in (polygon[0], polygon[i], polygon[i + 1]):
                    out_positions.append(positions[vi])
                    out_texcoords.append(texcoords[ti] if ti is not None else (0.0, 0.0))

    close_segment()

    vertex_data = np.zeros(
        len(out_positions),
        dtype=[("position", np.float32, 3), ("texcoord", np.float32, 2)],
    )
    vertex_data["position"] = out_positions
    vertex_data["texcoord"] = out_texcoords
    return CpuMesh(obj_path.stem, vertex_data, segments)

## 4. Catalogo da cena

Os modelos estao separados por funcao narrativa. `club_room`, `dance_floor`, `lounge_floor`, `parking_lot` e `night_skybox` formam os ambientes, pisos e ceu. Os objetos principais do ambiente interno e externo ficam em arquivos `.obj` separados e possuem textura de imagem via `map_Kd` ou fallback texturizado equivalente. Repeticoes de pessoas e postes sao apenas instancias do mesmo modelo, como permitido no enunciado. Mantive a area externa com modelos mais leves para evitar travamentos na renderizacao.

In [49]:
SCENE_CATALOG = {
    "ambiente": [
        "club_room",
        "lounge_floor",
        "dance_floor",
        "parking_lot",
        "night_skybox",
    ],
    "interno": [
        "party_person_01",
        "party_person_male_01",
        "boombox",
        "bar_chair_round_01",
        "sofa_01",
        "modern_ceiling_lamp_01",
        "disco_ball",
        "disco_support",
    ],
    "externo": [
        "rollershutter_door",
        "corrado_car_01",
        "street_lamp_01",
        "oga_trash_can_01",
        "party_person_01",
        "party_person_male_01",
    ],
}

for group, names in SCENE_CATALOG.items():
    print(f"{group}:")
    for model_name in names:
        mesh = load_obj_mesh(MODELS_DIR / f"{model_name}.obj")
        materials = ", ".join(dict.fromkeys(segment.material for segment in mesh.segments))
        textured_segments = sum(1 for segment in mesh.segments if segment.texture_path is not None)
        print(f"  - {model_name:36s} | vertices: {len(mesh.vertex_data):7d} | segmentos texturizados: {textured_segments:2d}/{len(mesh.segments):2d} | materiais: {materials}")

ambiente:
  - club_room                            | vertices:      42 | segmentos texturizados:  2/ 2 | materiais: club_wall, club_ceiling
  - lounge_floor                         | vertices:       6 | segmentos texturizados:  1/ 1 | materiais: lounge_floor_mat
  - dance_floor                          | vertices:       6 | segmentos texturizados:  1/ 1 | materiais: dance_floor_mat
  - parking_lot                          | vertices:       6 | segmentos texturizados:  1/ 1 | materiais: parking_lot_mat
  - night_skybox                         | vertices:    6912 | segmentos texturizados:  1/ 1 | materiais: sky_mat
interno:
  - party_person_01                      | vertices:   10488 | segmentos texturizados:  6/ 6 | materiais: Material__61, Material__62, Material__63, Material__60, Material__64, Material__65
  - party_person_male_01                 | vertices:    9000 | segmentos texturizados:  1/ 1 | materiais: CMan0010
  - boombox                              | vertices:   30504 | seg

## 5. Janela, contexto OpenGL e envio dos modelos para a GPU

Depois de criar a janela, cada modelo recebe um VAO/VBO proprio. Cada segmento do modelo sabe qual textura deve usar, de acordo com o material lido no `.mtl`.

In [50]:
WINDOW_WIDTH = 1000
WINDOW_HEIGHT = 720

if not glfw.init():
    raise RuntimeError("Nao foi possivel inicializar GLFW.")

glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)

window = glfw.create_window(WINDOW_WIDTH, WINDOW_HEIGHT, "Projeto 2 - Balada urbana", None, None)
if window is None:
    glfw.terminate()
    raise RuntimeError("Nao foi possivel criar a janela GLFW.")

glfw.make_context_current(window)
glfw.swap_interval(1)
glViewport(0, 0, WINDOW_WIDTH, WINDOW_HEIGHT)

glEnable(GL_DEPTH_TEST)
glDisable(GL_CULL_FACE)

shader = ShaderProgram(VERTEX_SHADER_SOURCE, FRAGMENT_SHADER_SOURCE)
shader.use()

glUniform1i(shader.uniform("imageTexture"), 0)

texture_cache = {}
solid_texture_cache = {}

def load_texture(texture_path):
    texture_path = Path(texture_path).resolve()
    if texture_path in texture_cache:
        return texture_cache[texture_path]

    image = Image.open(texture_path).transpose(Image.FLIP_TOP_BOTTOM).convert("RGBA")
    width, height = image.size
    pixels = image.tobytes()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, pixels)
    glGenerateMipmap(GL_TEXTURE_2D)

    texture_cache[texture_path] = texture_id
    return texture_id

def load_solid_color_texture(diffuse_color):
    color_key = tuple(round(float(c), 4) for c in diffuse_color)
    if color_key in solid_texture_cache:
        return solid_texture_cache[color_key]

    rgba = bytes([
        max(0, min(255, int(color_key[0] * 255))),
        max(0, min(255, int(color_key[1] * 255))),
        max(0, min(255, int(color_key[2] * 255))),
        255,
    ])
    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, 1, 1, 0, GL_RGBA, GL_UNSIGNED_BYTE, rgba)
    solid_texture_cache[color_key] = texture_id
    return texture_id

def upload_mesh(cpu_mesh):
    vao = glGenVertexArrays(1)
    vbo = glGenBuffers(1)

    glBindVertexArray(vao)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, cpu_mesh.vertex_data.nbytes, cpu_mesh.vertex_data, GL_STATIC_DRAW)

    stride = cpu_mesh.vertex_data.strides[0]
    pos_offset = ctypes.c_void_p(cpu_mesh.vertex_data.dtype.fields["position"][1])
    uv_offset = ctypes.c_void_p(cpu_mesh.vertex_data.dtype.fields["texcoord"][1])

    glEnableVertexAttribArray(0)
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, stride, pos_offset)
    glEnableVertexAttribArray(1)
    glVertexAttribPointer(1, 2, GL_FLOAT, GL_FALSE, stride, uv_offset)

    gpu_segments = []
    for segment in cpu_mesh.segments:
        gpu_segments.append({
            "start": segment.start,
            "count": segment.count,
            "material": segment.material,
            "texture_id": load_texture(segment.texture_path) if segment.texture_path else load_solid_color_texture(segment.diffuse_color),
        })

    glBindVertexArray(0)
    return GpuMesh(cpu_mesh.name, vao, vbo, gpu_segments, len(cpu_mesh.vertex_data))

all_model_names = list(dict.fromkeys(SCENE_CATALOG["ambiente"] + SCENE_CATALOG["interno"] + SCENE_CATALOG["externo"]))
gpu_meshes = {}
for model_name in all_model_names:
    cpu_mesh = load_obj_mesh(MODELS_DIR / f"{model_name}.obj")
    gpu_meshes[model_name] = upload_mesh(cpu_mesh)

print("Modelos enviados para a GPU:", ", ".join(gpu_meshes.keys()))

Modelos enviados para a GPU: club_room, lounge_floor, dance_floor, parking_lot, night_skybox, party_person_01, party_person_male_01, boombox, bar_chair_round_01, sofa_01, modern_ceiling_lamp_01, disco_ball, disco_support, rollershutter_door, corrado_car_01, street_lamp_01, oga_trash_can_01


## 6. Camera e controles

A exploracao fica limitada dentro dos limites do skybox e do terreno externo. A camera pode atravessar objetos, mas nao sai da area util da balada.

Controles principais:

- `W`, `A`, `S`, `D`: mover no plano horizontal.
- `Espaco` / `C`: subir e descer.
- Mouse: olhar ao redor.
- `P`: exibir/ocultar malha poligonal.
- `T` / `G`: transladar o carro na area externa.
- `R` / `F`: rotacionar a bola de discoteca.
- `Z` / `X`: alterar a escala do boombox no palco.
- `Esc`: fechar a janela.

In [51]:
camera_pos = glm.vec3(0.0, 1.7, 9.5)
camera_front = glm.normalize(glm.vec3(0.0, -0.10, -1.0))
camera_up = glm.vec3(0.0, 1.0, 0.0)

yaw = -90.0
pitch = -7.0
last_x = WINDOW_WIDTH / 2
last_y = WINDOW_HEIGHT / 2
first_mouse = True
fov = 45.0

show_wireframe = False
car_translation = 0.0
disco_rotation = 0.0
boombox_scale_delta = 0.0

SCENE_LIMIT_X = 17.5
SCENE_MIN_Z = -10.0
SCENE_MAX_Z = 31.0
MIN_CAMERA_Y = 0.35
MAX_CAMERA_Y = 9.0

def clamp(value, lower, upper):
    return max(lower, min(upper, value))

def clamp_camera_position():
    global camera_pos
    camera_pos.x = clamp(camera_pos.x, -SCENE_LIMIT_X, SCENE_LIMIT_X)
    camera_pos.y = clamp(camera_pos.y, MIN_CAMERA_Y, MAX_CAMERA_Y)
    camera_pos.z = clamp(camera_pos.z, SCENE_MIN_Z, SCENE_MAX_Z)

def key_callback(active_window, key, scancode, action, mods):
    global show_wireframe
    _ = scancode, mods
    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(active_window, True)
    if key == glfw.KEY_P and action == glfw.PRESS:
        show_wireframe = not show_wireframe
        print("Malha poligonal:", "visivel" if show_wireframe else "oculta")

def mouse_callback(_window, xpos, ypos):
    global yaw, pitch, last_x, last_y, first_mouse, camera_front
    if first_mouse:
        last_x = xpos
        last_y = ypos
        first_mouse = False

    xoffset = xpos - last_x
    yoffset = last_y - ypos
    last_x = xpos
    last_y = ypos

    sensitivity = 0.08
    yaw += xoffset * sensitivity
    pitch += yoffset * sensitivity
    pitch = clamp(pitch, -89.0, 89.0)

    direction = glm.vec3(
        math.cos(math.radians(yaw)) * math.cos(math.radians(pitch)),
        math.sin(math.radians(pitch)),
        math.sin(math.radians(yaw)) * math.cos(math.radians(pitch)),
    )
    camera_front = glm.normalize(direction)

def framebuffer_size_callback(_window, width, height):
    global WINDOW_WIDTH, WINDOW_HEIGHT
    WINDOW_WIDTH = max(1, width)
    WINDOW_HEIGHT = max(1, height)
    glViewport(0, 0, WINDOW_WIDTH, WINDOW_HEIGHT)

def process_input(delta_time):
    global camera_pos, car_translation, disco_rotation, boombox_scale_delta

    speed = 5.5 if glfw.get_key(window, glfw.KEY_LEFT_SHIFT) == glfw.PRESS else 3.2
    distance = speed * delta_time
    front_flat = glm.vec3(camera_front.x, 0.0, camera_front.z)
    if glm.length(front_flat) < 0.001:
        front_flat = glm.vec3(0.0, 0.0, -1.0)
    front_flat = glm.normalize(front_flat)
    right = glm.normalize(glm.cross(front_flat, camera_up))

    if glfw.get_key(window, glfw.KEY_W) == glfw.PRESS:
        camera_pos += front_flat * distance
    if glfw.get_key(window, glfw.KEY_S) == glfw.PRESS:
        camera_pos -= front_flat * distance
    if glfw.get_key(window, glfw.KEY_D) == glfw.PRESS:
        camera_pos += right * distance
    if glfw.get_key(window, glfw.KEY_A) == glfw.PRESS:
        camera_pos -= right * distance
    if glfw.get_key(window, glfw.KEY_SPACE) == glfw.PRESS:
        camera_pos += camera_up * distance
    if glfw.get_key(window, glfw.KEY_C) == glfw.PRESS:
        camera_pos -= camera_up * distance

    if glfw.get_key(window, glfw.KEY_T) == glfw.PRESS:
        car_translation = clamp(car_translation + 3.0 * delta_time, -7.0, 7.0)
    if glfw.get_key(window, glfw.KEY_G) == glfw.PRESS:
        car_translation = clamp(car_translation - 3.0 * delta_time, -7.0, 7.0)
    if glfw.get_key(window, glfw.KEY_R) == glfw.PRESS:
        disco_rotation += 110.0 * delta_time
    if glfw.get_key(window, glfw.KEY_F) == glfw.PRESS:
        disco_rotation -= 110.0 * delta_time
    if glfw.get_key(window, glfw.KEY_Z) == glfw.PRESS:
        boombox_scale_delta = clamp(boombox_scale_delta + 0.55 * delta_time, -0.35, 0.75)
    if glfw.get_key(window, glfw.KEY_X) == glfw.PRESS:
        boombox_scale_delta = clamp(boombox_scale_delta - 0.55 * delta_time, -0.35, 0.75)

    clamp_camera_position()

glfw.set_key_callback(window, key_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

## 7. Nos da cena e matriz Model

Cada item da cena tem posicao, rotacao e escala proprias. As transformacoes independentes exigidas pelo projeto foram aplicadas em modelos diferentes: translacao no carro externo, rotacao na bola de discoteca e escala no boombox. A pista e o salao foram separados para manter texturas diferentes no interior, e a area externa usa estacionamento, skybox e entrada metalica como delimitacao urbana sem contar esses elementos como os seis objetos principais.

In [55]:
def vec3_tuple(values):
    return glm.vec3(float(values[0]), float(values[1]), float(values[2]))

def node(mesh, position=(0, 0, 0), scale=(1, 1, 1), rotations=None):
    return {
        "mesh": mesh,
        "position": vec3_tuple(position),
        "scale": vec3_tuple(scale),
        "rotations": rotations or [],
    }

scene_nodes = [
    node("parking_lot", position=(0.0, -0.03, 18.0)),
    node("rollershutter_door", position=(-1.85, 0.02, 7.62), scale=(1.85, 1.55, 1.55)),
    node("club_room"),
    node("lounge_floor", position=(0.0, 0.005, 0.0), scale=(1.0, 1.0, 1.0)),
    node("dance_floor", position=(0.0, 0.02, 0.0), scale=(1.0, 1.0, 1.0)),

    node("party_person_01", position=(-2.1, 0.02, -0.8), scale=(0.92, 0.92, 0.92), rotations=[(188, (0, 1, 0))]),
    node("party_person_01", position=(1.8, 0.02, -0.5), scale=(0.90, 0.90, 0.90), rotations=[(170, (0, 1, 0))]),
    node("party_person_01", position=(0.7, 0.02, 1.2), scale=(0.91, 0.91, 0.91), rotations=[(176, (0, 1, 0))]),
    node("party_person_01", position=(-1.4, 0.02, -2.0), scale=(0.89, 0.89, 0.89), rotations=[(194, (0, 1, 0))]),
    node("party_person_01", position=(-5.6, 0.02, 3.8), scale=(0.88, 0.88, 0.88), rotations=[(-18, (0, 1, 0))]),
    node("party_person_01", position=(5.5, 0.02, 3.7), scale=(0.88, 0.88, 0.88), rotations=[(22, (0, 1, 0))]),
    node("party_person_01", position=(-6.9, 0.02, 1.3), scale=(0.86, 0.86, 0.86), rotations=[(-8, (0, 1, 0))]),
    node("party_person_01", position=(6.8, 0.02, 1.4), scale=(0.87, 0.87, 0.87), rotations=[(10, (0, 1, 0))]),
    node("party_person_01", position=(-3.4, 0.02, 4.8), scale=(0.87, 0.87, 0.87), rotations=[(146, (0, 1, 0))]),
    node("party_person_01", position=(3.5, 0.02, 4.9), scale=(0.87, 0.87, 0.87), rotations=[(-142, (0, 1, 0))]),

    node("party_person_male_01", position=(-0.6, 0.02, 0.6), scale=(0.96, 0.96, 0.96), rotations=[(4, (0, 1, 0))]),
    node("party_person_male_01", position=(-0.2, 0.02, -1.3), scale=(0.97, 0.97, 0.97), rotations=[(6, (0, 1, 0))]),
    node("party_person_male_01", position=(2.6, 0.02, -1.0), scale=(0.94, 0.94, 0.94), rotations=[(-14, (0, 1, 0))]),
    node("party_person_male_01", position=(1.5, 0.02, -2.5), scale=(0.95, 0.95, 0.95), rotations=[(-6, (0, 1, 0))]),
    node("party_person_male_01", position=(-4.8, 0.02, 2.7), scale=(0.92, 0.92, 0.92), rotations=[(34, (0, 1, 0))]),
    node("party_person_male_01", position=(4.9, 0.02, 2.8), scale=(0.92, 0.92, 0.92), rotations=[(-32, (0, 1, 0))]),
    node("party_person_male_01", position=(-4.9, 0.02, 5.1), scale=(0.90, 0.90, 0.90), rotations=[(40, (0, 1, 0))]),
    node("party_person_male_01", position=(5.0, 0.02, 5.0), scale=(0.90, 0.90, 0.90), rotations=[(-38, (0, 1, 0))]),

    node("boombox", position=(0.0, 0.02, -5.8), scale=(4.15, 4.15, 4.15), rotations=[]),

    node("sofa_01", position=(-8.55, 0.02, 4.5), scale=(2.1, 2.1, 2.1), rotations=[(90, (0, 1, 0))]),
    node("sofa_01", position=(8.55, 0.02, 4.5), scale=(2.1, 2.1, 2.1), rotations=[(-90, (0, 1, 0))]),
    node("bar_chair_round_01", position=(-8.7, 0.02, 1.9), scale=(1.26, 1.26, 1.26), rotations=[(90, (0, 1, 0))]),
    node("bar_chair_round_01", position=(-8.7, 0.02, 7.05), scale=(1.26, 1.26, 1.26), rotations=[(90, (0, 1, 0))]),
    node("bar_chair_round_01", position=(8.7, 0.02, 1.9), scale=(1.26, 1.26, 1.26), rotations=[(-90, (0, 1, 0))]),
    node("bar_chair_round_01", position=(8.7, 0.02, 7.05), scale=(1.26, 1.26, 1.26), rotations=[(-90, (0, 1, 0))]),

    node("modern_ceiling_lamp_01", position=(-6.5, 3.75, -1.4), scale=(1.45, 1.45, 1.45)),
    node("modern_ceiling_lamp_01", position=(-2.2, 3.75, -1.4), scale=(1.45, 1.45, 1.45)),
    node("modern_ceiling_lamp_01", position=(2.2, 3.75, -1.4), scale=(1.45, 1.45, 1.45)),
    node("modern_ceiling_lamp_01", position=(6.5, 3.75, -1.4), scale=(1.45, 1.45, 1.45)),
    node("modern_ceiling_lamp_01", position=(-4.2, 3.75, 2.4), scale=(1.3, 1.3, 1.3)),
    node("modern_ceiling_lamp_01", position=(0.0, 3.75, 2.4), scale=(1.3, 1.3, 1.3)),
    node("modern_ceiling_lamp_01", position=(4.2, 3.75, 2.4), scale=(1.3, 1.3, 1.3)),
    node("disco_support", position=(0.0, 4.15, -0.3), scale=(1.02, 1.25, 1.02)),
    node("disco_ball", position=(0.0, 3.68, -0.3), scale=(0.84, 0.84, 0.84)),

    node("corrado_car_01", position=(11.8, 0.03, 14.9), scale=(0.0092, 0.0092, 0.0092), rotations=[(0, (0, 1, 0))]),
    node("oga_trash_can_01", position=(-8.95, 0.03, 8.05), scale=(0.34, 0.34, 0.34), rotations=[(180, (0, 1, 0))]),

    node("party_person_01", position=(-5.2, 0.03, 12.8), scale=(0.82, 0.82, 0.82), rotations=[(30, (0, 1, 0))]),
    node("party_person_male_01", position=(-2.3, 0.03, 16.3), scale=(0.88, 0.88, 0.88), rotations=[(72, (0, 1, 0))]),
    node("party_person_01", position=(2.6, 0.03, 18.7), scale=(0.80, 0.80, 0.80), rotations=[(-36, (0, 1, 0))]),
    node("party_person_male_01", position=(8.8, 0.03, 19.6), scale=(0.84, 0.84, 0.84), rotations=[(-120, (0, 1, 0))]),
    node("party_person_01", position=(-11.8, 0.03, 20.8), scale=(0.78, 0.78, 0.78), rotations=[(118, (0, 1, 0))]),
    node("party_person_male_01", position=(12.0, 0.03, 22.1), scale=(0.82, 0.82, 0.82), rotations=[(-122, (0, 1, 0))]),

    node("street_lamp_01", position=(-16.0, 0.03, 10.5), scale=(1.38, 1.38, 1.38)),
    node("street_lamp_01", position=(-16.0, 0.03, 17.5), scale=(1.38, 1.38, 1.38)),
    node("street_lamp_01", position=(-16.0, 0.03, 24.5), scale=(1.38, 1.38, 1.38)),
    node("street_lamp_01", position=(16.0, 0.03, 10.5), scale=(1.38, 1.38, 1.38)),
    node("street_lamp_01", position=(16.0, 0.03, 17.5), scale=(1.38, 1.38, 1.38)),
    node("street_lamp_01", position=(16.0, 0.03, 24.5), scale=(1.38, 1.38, 1.38)),
]

def build_model_matrix(scene_node):
    model = glm.mat4(1.0)
    position = glm.vec3(scene_node["position"])
    scale = glm.vec3(scene_node["scale"])

    if scene_node["mesh"] == "corrado_car_01":
        position += glm.vec3(car_translation, 0.0, 0.0)
    if scene_node["mesh"] == "boombox":
        factor = 1.0 + boombox_scale_delta
        scale *= glm.vec3(factor, factor, factor)

    model = glm.translate(model, position)

    if scene_node["mesh"] == "disco_ball":
        model = glm.rotate(model, glm.radians(disco_rotation), glm.vec3(0.0, 1.0, 0.0))

    for angle_degrees, axis in scene_node["rotations"]:
        model = glm.rotate(model, glm.radians(angle_degrees), vec3_tuple(axis))

    model = glm.scale(model, scale)
    return model

def view_matrix():
    return glm.lookAt(camera_pos, camera_pos + camera_front, camera_up)

def projection_matrix():
    aspect = WINDOW_WIDTH / WINDOW_HEIGHT
    return glm.perspective(glm.radians(fov), aspect, 0.1, 120.0)


## 8. Desenho texturizado e visualizacao da malha

O desenho normal usa as texturas dos materiais. Quando `P` e pressionado, os mesmos modelos sao redesenhados com `glPolygonMode(GL_LINE)` e uma cor solida, permitindo ver a malha poligonal sem trocar os arquivos de modelo.

In [53]:
loc_model = shader.uniform("model")
loc_view = shader.uniform("view")
loc_projection = shader.uniform("projection")
loc_use_solid_color = shader.uniform("useSolidColor")
loc_solid_color = shader.uniform("solidColor")

def send_matrix(location, matrix):
    glUniformMatrix4fv(location, 1, GL_FALSE, glm.value_ptr(matrix))

def draw_gpu_mesh(gpu_mesh):
    glBindVertexArray(gpu_mesh.vao)
    for segment in gpu_mesh.segments:
        glActiveTexture(GL_TEXTURE0)
        glBindTexture(GL_TEXTURE_2D, segment["texture_id"])
        glDrawArrays(GL_TRIANGLES, segment["start"], segment["count"])
    glBindVertexArray(0)

def draw_node(scene_node):
    send_matrix(loc_model, build_model_matrix(scene_node))
    draw_gpu_mesh(gpu_meshes[scene_node["mesh"]])

def draw_scene_nodes(wireframe=False):
    if wireframe:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
        glUniform1i(loc_use_solid_color, GL_TRUE)
        glUniform4f(loc_solid_color, 0.02, 0.02, 0.02, 1.0)
        glLineWidth(1.0)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)
        glUniform1i(loc_use_solid_color, GL_FALSE)

    for item in scene_nodes:
        draw_node(item)

    glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

## 9. Loop principal

Execute esta celula para abrir a janela. Para encerrar, pressione `Esc` ou feche a janela. Se o cursor ficar preso na janela por causa do modo de camera, `Esc` tambem encerra a execucao.

In [54]:
last_frame = glfw.get_time()

while not glfw.window_should_close(window):
    current_frame = glfw.get_time()
    delta_time = current_frame - last_frame
    last_frame = current_frame

    glfw.poll_events()
    process_input(delta_time)

    glClearColor(0.015, 0.015, 0.022, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    shader.use()
    send_matrix(loc_view, view_matrix())
    send_matrix(loc_projection, projection_matrix())

    # Skybox texturizado. Ele delimita visualmente a borda da exploracao.
    glDepthMask(GL_FALSE)
    glUniform1i(loc_use_solid_color, GL_FALSE)
    sky_model = glm.translate(glm.mat4(1.0), camera_pos)
    sky_model = glm.scale(sky_model, glm.vec3(60.0, 60.0, 60.0))
    send_matrix(loc_model, sky_model)
    draw_gpu_mesh(gpu_meshes["night_skybox"])
    glDepthMask(GL_TRUE)

    draw_scene_nodes(wireframe=False)
    if show_wireframe:
        draw_scene_nodes(wireframe=True)

    glfw.swap_buffers(window)

glfw.terminate()
print("Janela encerrada.")

Janela encerrada.


## 10. Checklist dos requisitos atendidos

- Ambiente interno: clube/salao modelado em `.obj` (`club_room`) e fechado por paredes texturizadas.
- Ambiente externo: rua/estacionamento com piso de asfalto, entrada metalica, postes alinhados, carro, lixeira e pessoas.
- Seis ou mais modelos 3D texturizados, sem contar os delimitadores dos ambientes: pessoas, boombox, bancos, sofas, luminarias, bola de discoteca, carro, lixeira e postes.
- Pelo menos tres modelos no ambiente interno: pessoas, boombox, bancos/sofas, luminarias e bola de discoteca.
- Pelo menos tres modelos no ambiente externo: carro, lixeira, postes e pessoas.
- Cada item principal vem de um arquivo `.obj` diferente; repeticoes sao apenas instancias do mesmo modelo.
- Piso interno com pista quadrada central texturizada (`dance_floor`) e piso restante em grafite frio (`lounge_floor`), diferente do piso externo de asfalto (`parking_lot`).
- Skybox com textura de ceu noturno e limites de exploracao da camera dentro do terreno/ceu.
- Toggle de malha poligonal com `P`.
- Transformacoes independentes por teclado: translacao no carro, rotacao na bola de discoteca e escala no boombox.
- Matriz `model` por objeto, matriz `view` pela camera e matriz `projection` em perspectiva.
- Sem iluminacao: o fragment shader usa diretamente textura difusa ou cor solida apenas no overlay da malha.
- Sem funcoes depreciadas do pipeline fixo (`glBegin`, `glEnd`, `glTranslate`, `glRotate`, `glScale`, `glLight`, `glMaterial`, matrizes antigas etc.).